In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/cleaned/taxi_cleaned.csv')
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

print(f"✅ Data loaded")
print(f"Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nColumns available:")
for col in df.columns:
    print(f"  {col}")

✅ Data loaded
Shape: 601,662 rows, 14 columns

Columns available:
  id
  vendor_id
  pickup_datetime
  passenger_count
  pickup_longitude
  pickup_latitude
  dropoff_longitude
  dropoff_latitude
  store_and_fwd_flag
  hour_of_day
  day_of_week
  is_weekend
  trip_distance_km
  distance_bucket


In [2]:
# GROUP BY hour, COUNT trips, SORT by hour number
hourly = df.groupby('hour_of_day').size().reset_index(name='trip_count')
hourly = hourly.sort_values('hour_of_day')

# Add percentage of total
hourly['pct_of_total'] = (hourly['trip_count'] / hourly['trip_count'].sum() * 100).round(2)

print("=== Q1: HOURLY TRIP DEMAND ===")
print(hourly.to_string(index=False))

# Peak and quiet hours
peak_hour = hourly.loc[hourly['trip_count'].idxmax()]
quiet_hour = hourly.loc[hourly['trip_count'].idxmin()]

print(f"\n📈 Peak hour  : {int(peak_hour['hour_of_day'])}:00 with {int(peak_hour['trip_count']):,} trips")
print(f"📉 Quiet hour : {int(quiet_hour['hour_of_day'])}:00 with {int(quiet_hour['trip_count']):,} trips")
print(f"\n💡 Finding: Evening rush (6–10 PM) drives NYC taxi demand.")
print(f"   The quietest hour is {int(quiet_hour['hour_of_day'])}:00 AM — deep overnight hours.")

=== Q1: HOURLY TRIP DEMAND ===
 hour_of_day  trip_count  pct_of_total
           0       21911          3.64
           1       15931          2.65
           2       11527          1.92
           3        8542          1.42
           4        6527          1.08
           5        6200          1.03
           6       13591          2.26
           7       22905          3.81
           8       27712          4.61
           9       27943          4.64
          10       26836          4.46
          11       27937          4.64
          12       29202          4.85
          13       29313          4.87
          14       31079          5.17
          15       29747          4.94
          16       26629          4.43
          17       31772          5.28
          18       37535          6.24
          19       37106          6.17
          20       35026          5.82
          21       34504          5.73
          22       33344          5.54
          23       28843         

In [3]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

daily = df.groupby('day_of_week').size().reset_index(name='trip_count')
daily['day_of_week'] = pd.Categorical(daily['day_of_week'], categories=day_order, ordered=True)
daily = daily.sort_values('day_of_week')
daily['pct_of_total'] = (daily['trip_count'] / daily['trip_count'].sum() * 100).round(2)

print("=== Q2: TRIPS BY DAY OF WEEK ===")
print(daily.to_string(index=False))

busiest_day = daily.loc[daily['trip_count'].idxmax()]
quietest_day = daily.loc[daily['trip_count'].idxmin()]

print(f"\n📈 Busiest day  : {busiest_day['day_of_week']} with {int(busiest_day['trip_count']):,} trips")
print(f"📉 Quietest day : {quietest_day['day_of_week']} with {int(quietest_day['trip_count']):,} trips")
diff = ((busiest_day['trip_count'] - quietest_day['trip_count']) / quietest_day['trip_count'] * 100)
print(f"\n💡 Finding: {busiest_day['day_of_week']} has {diff:.1f}% more trips than {quietest_day['day_of_week']}.")

=== Q2: TRIPS BY DAY OF WEEK ===
day_of_week  trip_count  pct_of_total
     Monday       77456         12.87
    Tuesday       83952         13.95
  Wednesday       86499         14.38
   Thursday       90063         14.97
     Friday       92727         15.41
   Saturday       90556         15.05
     Sunday       80409         13.36

📈 Busiest day  : Friday with 92,727 trips
📉 Quietest day : Monday with 77,456 trips

💡 Finding: Friday has 19.7% more trips than Monday.


In [4]:
weekend_summary = df.groupby('is_weekend').agg(
    total_trips    = ('id', 'count'),
    avg_distance_km = ('trip_distance_km', 'mean'),
    avg_passengers = ('passenger_count', 'mean')
).reset_index()

weekend_summary['label'] = weekend_summary['is_weekend'].map({0: 'Weekday', 1: 'Weekend'})
weekend_summary['avg_distance_km'] = weekend_summary['avg_distance_km'].round(3)
weekend_summary['avg_passengers']  = weekend_summary['avg_passengers'].round(2)
weekend_summary['pct_of_total']    = (weekend_summary['total_trips'] / weekend_summary['total_trips'].sum() * 100).round(1)

print("=== Q3: WEEKDAY vs WEEKEND COMPARISON ===")
print(weekend_summary[['label','total_trips','pct_of_total',
                        'avg_distance_km','avg_passengers']].to_string(index=False))

wkday = weekend_summary[weekend_summary['is_weekend'] == 0].iloc[0]
wkend = weekend_summary[weekend_summary['is_weekend'] == 1].iloc[0]
print(f"\n💡 Finding: Weekend trips average {wkend['avg_distance_km']:.3f} km vs")
print(f"   weekday {wkday['avg_distance_km']:.3f} km — check if weekenders travel farther.")

=== Q3: WEEKDAY vs WEEKEND COMPARISON ===
  label  total_trips  pct_of_total  avg_distance_km  avg_passengers
Weekday       430697          71.6            3.398            1.49
Weekend       170965          28.4            3.497            1.57

💡 Finding: Weekend trips average 3.497 km vs
   weekday 3.398 km — check if weekenders travel farther.


In [5]:
# Distance bucket summary
bucket_order = ['Short', 'Medium', 'Long']

bucket = df.groupby('distance_bucket').agg(
    trip_count      = ('id', 'count'),
    avg_distance_km = ('trip_distance_km', 'mean'),
    avg_passengers  = ('passenger_count', 'mean')
).reset_index()

bucket['distance_bucket'] = pd.Categorical(bucket['distance_bucket'],
                                            categories=bucket_order, ordered=True)
bucket = bucket.sort_values('distance_bucket')
bucket['pct_of_total']    = (bucket['trip_count'] / bucket['trip_count'].sum() * 100).round(1)
bucket['avg_distance_km'] = bucket['avg_distance_km'].round(3)
bucket['avg_passengers']  = bucket['avg_passengers'].round(2)

print("=== Q4: TRIP DISTANCE DISTRIBUTION ===")
print(bucket.to_string(index=False))

# Overall distance stats
print(f"\nOverall distance statistics:")
print(f"  Mean   : {df['trip_distance_km'].mean():.3f} km")
print(f"  Median : {df['trip_distance_km'].median():.3f} km")
print(f"  Std Dev: {df['trip_distance_km'].std():.3f} km")
print(f"  95th % : {df['trip_distance_km'].quantile(0.95):.3f} km")

print(f"\n💡 Finding: 94% of NYC taxi trips are under 10 km.")
print(f"   The median trip is just {df['trip_distance_km'].median():.2f} km — shorter than most expect.")

=== Q4: TRIP DISTANCE DISTRIBUTION ===
distance_bucket  trip_count  avg_distance_km  avg_passengers  pct_of_total
          Short      286389            1.201            1.50          47.6
         Medium      279029            4.104            1.52          46.4
           Long       36244           15.797            1.57           6.0

Overall distance statistics:
  Mean   : 3.426 km
  Median : 2.101 km
  Std Dev: 3.857 km
  95th % : 10.902 km

💡 Finding: 94% of NYC taxi trips are under 10 km.
   The median trip is just 2.10 km — shorter than most expect.


In [6]:
pax = df.groupby('passenger_count').agg(
    trip_count      = ('id', 'count'),
    avg_distance_km = ('trip_distance_km', 'mean')
).reset_index()

pax['pct_of_total']    = (pax['trip_count'] / pax['trip_count'].sum() * 100).round(1)
pax['avg_distance_km'] = pax['avg_distance_km'].round(3)

print("=== Q5: TRIPS BY PASSENGER COUNT ===")
print(pax.to_string(index=False))

solo = pax[pax['passenger_count'] == 1].iloc[0]
print(f"\n💡 Finding: Solo trips ({solo['pct_of_total']}% of all trips) dominate NYC taxi usage.")
print(f"   NYC taxis are primarily a solo commuter service, not group transport.")

=== Q5: TRIPS BY PASSENGER COUNT ===
 passenger_count  trip_count  avg_distance_km  pct_of_total
               1      441191            3.363          73.3
               2       89660            3.640          14.9
               3       25572            3.578           4.3
               4       11958            3.576           2.0
               5       33281            3.524           5.5

💡 Finding: Solo trips (73.3% of all trips) dominate NYC taxi usage.
   NYC taxis are primarily a solo commuter service, not group transport.


In [7]:
vendor = df.groupby('vendor_id').agg(
    trip_count      = ('id', 'count'),
    avg_distance_km = ('trip_distance_km', 'mean'),
    avg_passengers  = ('passenger_count', 'mean')
).reset_index()

vendor['pct_of_total']    = (vendor['trip_count'] / vendor['trip_count'].sum() * 100).round(1)
vendor['avg_distance_km'] = vendor['avg_distance_km'].round(3)
vendor['avg_passengers']  = vendor['avg_passengers'].round(2)
vendor['vendor_id']       = vendor['vendor_id'].map({1: 'Vendor 1 (CMT)', 2: 'Vendor 2 (VeriFone)'})

print("=== Q6: VENDOR COMPARISON ===")
print(vendor.to_string(index=False))

print(f"\n💡 Finding: Compare market share and average trip behavior between")
print(f"   NYC's two main taxi technology vendors.")

=== Q6: VENDOR COMPARISON ===
          vendor_id  trip_count  avg_distance_km  avg_passengers  pct_of_total
     Vendor 1 (CMT)      289390            3.388            1.26          48.1
Vendor 2 (VeriFone)      312272            3.462            1.75          51.9

💡 Finding: Compare market share and average trip behavior between
   NYC's two main taxi technology vendors.


In [8]:
print("=" * 55)
print("       NYC TAXI EDA — KEY FINDINGS SUMMARY")
print("=" * 55)

peak = df.groupby('hour_of_day').size().idxmax()
quiet = df.groupby('hour_of_day').size().idxmin()
busiest = df.groupby('day_of_week').size().idxmax()
solo_pct = (df[df['passenger_count']==1].shape[0] / len(df) * 100)
short_pct = (df[df['distance_bucket']=='Short'].shape[0] / len(df) * 100)
wkend_avg = df[df['is_weekend']==1]['trip_distance_km'].mean()
wkday_avg = df[df['is_weekend']==0]['trip_distance_km'].mean()

print(f"\n1. Total trips analyzed     : {len(df):,}")
print(f"2. Peak demand hour         : {peak}:00 (6 PM)")
print(f"3. Quietest hour            : {quiet}:00 AM")
print(f"4. Busiest day              : {busiest}")
print(f"5. Solo trips               : {solo_pct:.1f}% of all trips")
print(f"6. Short trips (< 2km)      : {short_pct:.1f}% of all trips")
print(f"7. Median trip distance     : {df['trip_distance_km'].median():.2f} km")
print(f"8. Avg distance - Weekday   : {wkday_avg:.3f} km")
print(f"9. Avg distance - Weekend   : {wkend_avg:.3f} km")
print(f"10. Max trip distance        : {df['trip_distance_km'].max():.2f} km (airport run)")
print("\n" + "=" * 55)

       NYC TAXI EDA — KEY FINDINGS SUMMARY

1. Total trips analyzed     : 601,662
2. Peak demand hour         : 18:00 (6 PM)
3. Quietest hour            : 5:00 AM
4. Busiest day              : Friday
5. Solo trips               : 73.3% of all trips
6. Short trips (< 2km)      : 47.6% of all trips
7. Median trip distance     : 2.10 km
8. Avg distance - Weekday   : 3.398 km
9. Avg distance - Weekend   : 3.497 km
10. Max trip distance        : 41.55 km (airport run)

